In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/playground-series-s6e3/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e3/train.csv
/kaggle/input/competitions/playground-series-s6e3/test.csv
/kaggle/input/datasets/blastchar/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

# ── Load Data ─────────────────────────────────────────
train = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/train.csv')
test  = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/test.csv')
sub   = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/sample_submission.csv')

# ── Load Original Dataset ─────────────────────────────
try:
    orig = pd.read_csv('/kaggle/input/datasets/blastchar/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv')
    orig = orig.drop(columns=['customerID'], errors='ignore')
    orig['TotalCharges'] = pd.to_numeric(orig['TotalCharges'], errors='coerce')
    orig['Churn']        = orig['Churn'].map({'Yes': 1, 'No': 0})
    orig['id']           = -1
    print(f'✅ Original dataset loaded: {orig.shape}')
except:
    orig = None
    print('⚠️ Original dataset not found')

# ── Fix Target ─────────────────────────────────────────
if train['Churn'].dtype == object:
    train['Churn'] = train['Churn'].map({'Yes': 1, 'No': 0})
train['Churn'] = train['Churn'].astype(int)

# ── Merge Original ─────────────────────────────────────
if orig is not None:
    orig_cols = [c for c in train.columns if c in orig.columns]
    train     = pd.concat([train, orig[orig_cols]], axis=0, ignore_index=True)
    print(f'✅ Train after merge: {train.shape}')

print(f'Train : {train.shape} | Test : {test.shape}')
print(f'Churn balance:\n{train["Churn"].value_counts(normalize=True).round(3)}')

# ── Constants ──────────────────────────────────────────
TARGET   = 'Churn'
ID_COL   = 'id'
N_SPLITS = 10
SEED     = 42
skf      = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

# ── Feature Engineering ────────────────────────────────
def feature_engineering(df):
    df = df.copy()

    # Encode object cols
    cat_cols = df.select_dtypes(include='object').columns.tolist()
    cat_cols = [c for c in cat_cols if c not in [ID_COL, TARGET]]
    for col in cat_cols:
        df[col] = LabelEncoder().fit_transform(df[col].astype(str))

    # Force numeric
    for col in df.columns:
        if col in [ID_COL, TARGET]: continue
        df[col] = pd.to_numeric(df[col], errors='coerce')

    # Fill NaN
    df = df.fillna(df.median(numeric_only=True))

    # Interactions
    if 'tenure' in df.columns and 'MonthlyCharges' in df.columns:
        df['charge_per_tenure'] = df['MonthlyCharges'] / (df['tenure'] + 1)
        df['total_charge_est']  = df['MonthlyCharges'] * df['tenure']
        df['tenure_x_monthly']  = df['tenure'] * df['MonthlyCharges']

    if 'TotalCharges' in df.columns and 'MonthlyCharges' in df.columns:
        df['charge_ratio']  = df['TotalCharges'] / (df['MonthlyCharges'] + 1)
        df['charges_diff']  = df['TotalCharges'] - df['MonthlyCharges'] * df['tenure'] \
                              if 'tenure' in df.columns else df['TotalCharges'] - df['MonthlyCharges']

    if 'tenure' in df.columns:
        df['tenure_squared']   = df['tenure'] ** 2
        df['tenure_log']       = np.log1p(df['tenure'])
        df['is_new_customer']  = (df['tenure'] <= 3).astype(int)
        df['is_long_customer'] = (df['tenure'] >= 48).astype(int)

    if 'MonthlyCharges' in df.columns:
        df['monthly_log']   = np.log1p(df['MonthlyCharges'])
        df['is_high_payer'] = (df['MonthlyCharges'] > 65).astype(int)

    if 'TotalCharges' in df.columns:
        df['total_log'] = np.log1p(df['TotalCharges'])

    # Service count
    service_cols = [c for c in df.columns if c in [
        'PhoneService','MultipleLines','InternetService',
        'OnlineSecurity','OnlineBackup','DeviceProtection',
        'TechSupport','StreamingTV','StreamingMovies'
    ]]
    if service_cols:
        df['num_services'] = df[service_cols].sum(axis=1)

    # Aggregate stats
    num_cols = df.select_dtypes(include=np.number).columns.tolist()
    num_cols = [c for c in num_cols if c not in [ID_COL, TARGET]]
    df['num_mean'] = df[num_cols].mean(axis=1)
    df['num_std']  = df[num_cols].std(axis=1)
    df['num_max']  = df[num_cols].max(axis=1)
    df['num_min']  = df[num_cols].min(axis=1)

    return df

train_fe = feature_engineering(train)
test_fe  = feature_engineering(test)

FEATURES = [c for c in train_fe.columns if c not in [ID_COL, TARGET]]
X        = train_fe[FEATURES]
y        = train_fe[TARGET]
X_test   = test_fe[FEATURES].reindex(columns=FEATURES, fill_value=0)

print(f'\n✅ Features : {len(FEATURES)}')
print(f'X      : {X.shape}')
print(f'X_test : {X_test.shape}')

✅ Original dataset loaded: (7043, 21)
✅ Train after merge: (601237, 21)
Train : (601237, 21) | Test : (254655, 20)
Churn balance:
Churn
0    0.774
1    0.226
Name: proportion, dtype: float64

✅ Features : 36
X      : (601237, 36)
X_test : (254655, 36)


In [3]:
from xgboost  import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
import lightgbm as lgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

oof_xgb  = np.zeros(len(X))
oof_lgb  = np.zeros(len(X))
oof_cat  = np.zeros(len(X))
pred_xgb = np.zeros(len(X_test))
pred_lgb = np.zeros(len(X_test))
pred_cat = np.zeros(len(X_test))

# ── Optuna Tune XGBoost (3-fold quick CV) ─────────────
print('🔍 Tuning XGBoost with Optuna (50 trials)...')

def tune_xgb(trial):
    params = dict(
        n_estimators          = trial.suggest_int('n_estimators', 500, 2000),
        learning_rate         = trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        max_depth             = trial.suggest_int('max_depth', 3, 8),
        min_child_weight      = trial.suggest_int('min_child_weight', 1, 20),
        gamma                 = trial.suggest_float('gamma', 0.0, 1.0),
        subsample             = trial.suggest_float('subsample', 0.5, 1.0),
        colsample_bytree      = trial.suggest_float('colsample_bytree', 0.5, 1.0),
        reg_alpha             = trial.suggest_float('reg_alpha', 0.0, 2.0),
        reg_lambda            = trial.suggest_float('reg_lambda', 0.0, 4.0),
        scale_pos_weight      = (y==0).sum() / (y==1).sum(),
        tree_method           = 'hist',
        device                = 'cuda',
        eval_metric           = 'auc',
        early_stopping_rounds = 50,
        random_state          = SEED,
        n_jobs                = -1
    )
    cv   = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
    aucs = []
    for tr_idx, val_idx in cv.split(X, y):
        m = XGBClassifier(**params)
        m.fit(X.iloc[tr_idx], y.iloc[tr_idx],
              eval_set=[(X.iloc[val_idx], y.iloc[val_idx])],
              verbose=False)
        aucs.append(roc_auc_score(y.iloc[val_idx],
                    m.predict_proba(X.iloc[val_idx])[:, 1]))
    return np.mean(aucs)

study_xgb = optuna.create_study(direction='maximize')
study_xgb.optimize(tune_xgb, n_trials=50, show_progress_bar=True)

best_xgb_params = study_xgb.best_params
best_xgb_params.update(dict(
    scale_pos_weight      = (y==0).sum() / (y==1).sum(),
    tree_method           = 'hist',
    device                = 'cuda',
    eval_metric           = 'auc',
    early_stopping_rounds = 100,
    random_state          = SEED,
    n_jobs                = -1
))
print(f'\n✅ Best XGBoost AUC (3-fold): {study_xgb.best_value:.5f}')
print(f'Best params: {study_xgb.best_params}')

# ── Optuna Tune CatBoost (3-fold quick CV) ────────────
print('\n🔍 Tuning CatBoost with Optuna (30 trials)...')

def tune_cat(trial):
    params = dict(
        iterations         = trial.suggest_int('iterations', 500, 2000),
        learning_rate      = trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        depth              = trial.suggest_int('depth', 3, 8),
        l2_leaf_reg        = trial.suggest_float('l2_leaf_reg', 1.0, 10.0),
        bootstrap_type     = 'Bernoulli',
        subsample          = trial.suggest_float('subsample', 0.5, 1.0),
        min_data_in_leaf   = trial.suggest_int('min_data_in_leaf', 10, 100),
        auto_class_weights = 'Balanced',
        eval_metric        = 'AUC',
        task_type          = 'GPU',
        random_seed        = SEED,
        verbose            = False
    )
    cv   = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
    aucs = []
    for tr_idx, val_idx in cv.split(X, y):
        m = CatBoostClassifier(**params)
        m.fit(X.iloc[tr_idx], y.iloc[tr_idx],
              eval_set              = (X.iloc[val_idx], y.iloc[val_idx]),
              early_stopping_rounds = 50)
        aucs.append(roc_auc_score(y.iloc[val_idx],
                    m.predict_proba(X.iloc[val_idx])[:, 1]))
    return np.mean(aucs)

study_cat = optuna.create_study(direction='maximize')
study_cat.optimize(tune_cat, n_trials=30, show_progress_bar=True)

best_cat_params = study_cat.best_params
best_cat_params.update(dict(
    bootstrap_type     = 'Bernoulli',
    auto_class_weights = 'Balanced',
    eval_metric        = 'AUC',
    task_type          = 'GPU',
    random_seed        = SEED,
    verbose            = False
))
print(f'\n✅ Best CatBoost AUC (3-fold): {study_cat.best_value:.5f}')
print(f'Best params: {study_cat.best_params}')

🔍 Tuning XGBoost with Optuna (50 trials)...


  0%|          | 0/50 [00:00<?, ?it/s]


✅ Best XGBoost AUC (3-fold): 0.91542
Best params: {'n_estimators': 1175, 'learning_rate': 0.03520271086971772, 'max_depth': 5, 'min_child_weight': 14, 'gamma': 0.5416248427320737, 'subsample': 0.9465544689459167, 'colsample_bytree': 0.524819891325854, 'reg_alpha': 1.809104071224633, 'reg_lambda': 0.382495492495977}

🔍 Tuning CatBoost with Optuna (30 trials)...


  0%|          | 0/30 [00:00<?, ?it/s]

Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric perio


✅ Best CatBoost AUC (3-fold): 0.91514
Best params: {'iterations': 1990, 'learning_rate': 0.09978148009586234, 'depth': 3, 'l2_leaf_reg': 2.600801668915584, 'subsample': 0.8196454716299668, 'min_data_in_leaf': 72}


In [4]:
# ── Train XGBoost (tuned) ─────────────────────────────
print('\n' + '='*50)
print('🚀 Training XGBoost (tuned 10-fold)...')
print('='*50)

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    model = XGBClassifier(**best_xgb_params)
    model.fit(X_tr, y_tr,
              eval_set=[(X_val, y_val)],
              verbose=False)

    oof_xgb[val_idx]  = model.predict_proba(X_val)[:, 1]
    pred_xgb         += model.predict_proba(X_test)[:, 1] / N_SPLITS
    print(f'  Fold {fold+1:2d} | AUC: {roc_auc_score(y_val, oof_xgb[val_idx]):.5f}')

print(f'\n📊 XGBoost OOF AUC: {roc_auc_score(y, oof_xgb):.5f}')

# ── Train LightGBM ────────────────────────────────────
print('\n' + '='*50)
print('🚀 Training LightGBM (GPU)...')
print('='*50)

lgb_params = dict(
    n_estimators      = 2000,
    learning_rate     = 0.03,
    num_leaves        = 63,
    max_depth         = -1,
    min_child_samples = 20,
    feature_fraction  = 0.8,
    bagging_fraction  = 0.8,
    bagging_freq      = 5,
    reg_alpha         = 0.1,
    reg_lambda        = 1.0,
    is_unbalance      = True,
    device            = 'gpu',
    gpu_platform_id   = 0,
    gpu_device_id     = 0,
    random_state      = SEED,
    n_jobs            = -1,
    verbosity         = -1
)

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    model = LGBMClassifier(**lgb_params)
    model.fit(X_tr, y_tr,
              eval_set    = [(X_val, y_val)],
              eval_metric = 'auc',
              callbacks   = [
                  lgb.early_stopping(100, verbose=False),
                  lgb.log_evaluation(False)
              ])

    oof_lgb[val_idx]  = model.predict_proba(X_val)[:, 1]
    pred_lgb         += model.predict_proba(X_test)[:, 1] / N_SPLITS
    print(f'  Fold {fold+1:2d} | AUC: {roc_auc_score(y_val, oof_lgb[val_idx]):.5f}')

print(f'\n📊 LightGBM OOF AUC: {roc_auc_score(y, oof_lgb):.5f}')

# ── Train CatBoost (tuned) ────────────────────────────
print('\n' + '='*50)
print('🚀 Training CatBoost (tuned 10-fold)...')
print('='*50)

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    model = CatBoostClassifier(**best_cat_params)
    model.fit(X_tr, y_tr,
              eval_set              = (X_val, y_val),
              early_stopping_rounds = 100)

    oof_cat[val_idx]  = model.predict_proba(X_val)[:, 1]
    pred_cat         += model.predict_proba(X_test)[:, 1] / N_SPLITS
    print(f'  Fold {fold+1:2d} | AUC: {roc_auc_score(y_val, oof_cat[val_idx]):.5f}')

print(f'\n📊 CatBoost OOF AUC: {roc_auc_score(y, oof_cat):.5f}')

# ── Summary ───────────────────────────────────────────
print('\n' + '='*50)
print('📊 Model Summary')
print('='*50)
print(f'  XGBoost  OOF AUC : {roc_auc_score(y, oof_xgb):.5f}')
print(f'  LightGBM OOF AUC : {roc_auc_score(y, oof_lgb):.5f}')
print(f'  CatBoost OOF AUC : {roc_auc_score(y, oof_cat):.5f}')


🚀 Training XGBoost (tuned 10-fold)...
  Fold  1 | AUC: 0.91561
  Fold  2 | AUC: 0.91637
  Fold  3 | AUC: 0.91852
  Fold  4 | AUC: 0.91462
  Fold  5 | AUC: 0.91494
  Fold  6 | AUC: 0.91378
  Fold  7 | AUC: 0.91554
  Fold  8 | AUC: 0.91531
  Fold  9 | AUC: 0.91638
  Fold 10 | AUC: 0.91631

📊 XGBoost OOF AUC: 0.91573

🚀 Training LightGBM (GPU)...


1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


  Fold  1 | AUC: 0.91164
  Fold  2 | AUC: 0.91238
  Fold  3 | AUC: 0.91410
  Fold  4 | AUC: 0.90976
  Fold  5 | AUC: 0.91055
  Fold  6 | AUC: 0.90971
  Fold  7 | AUC: 0.91135
  Fold  8 | AUC: 0.91087
  Fold  9 | AUC: 0.91222
  Fold 10 | AUC: 0.91120

📊 LightGBM OOF AUC: 0.91129

🚀 Training CatBoost (tuned 10-fold)...


Default metric period is 5 because AUC is/are not implemented for GPU


  Fold  1 | AUC: 0.91523


...

  Fold  8 | AUC: 0.91516


Default metric period is 5 because AUC is/are not implemented for GPU


  Fold  9 | AUC: 0.91604


Default metric period is 5 because AUC is/are not implemented for GPU


  Fold 10 | AUC: 0.91599

📊 CatBoost OOF AUC: 0.91538

📊 Model Summary
  XGBoost  OOF AUC : 0.91573
  LightGBM OOF AUC : 0.91129
  CatBoost OOF AUC : 0.91538


In [5]:
# ── Pseudo-labeling ───────────────────────────────────
# Use high-confidence test predictions as extra training data
print('🔍 Generating pseudo-labels...')

# Ensemble current predictions for pseudo-labels
pseudo_probs = 0.5 * pred_xgb + 0.5 * pred_cat

# Only use HIGH CONFIDENCE predictions (far from 0.5)
THRESHOLD_HIGH = 0.85   # confident churn
THRESHOLD_LOW  = 0.10   # confident no churn

high_conf_mask = (pseudo_probs >= THRESHOLD_HIGH) | (pseudo_probs <= THRESHOLD_LOW)
pseudo_labels  = (pseudo_probs >= THRESHOLD_HIGH).astype(int)

print(f'Total test samples      : {len(pseudo_probs)}')
print(f'High confidence samples : {high_conf_mask.sum()}')
print(f'  → Pseudo churn=1      : {(pseudo_labels[high_conf_mask]==1).sum()}')
print(f'  → Pseudo churn=0      : {(pseudo_labels[high_conf_mask]==0).sum()}')

# Build augmented training set
X_pseudo    = X_test[high_conf_mask].copy()
y_pseudo    = pd.Series(pseudo_labels[high_conf_mask], name=TARGET)

X_augmented = pd.concat([X, X_pseudo],    axis=0, ignore_index=True)
y_augmented = pd.concat([y, y_pseudo],    axis=0, ignore_index=True)

print(f'\nOriginal train size  : {len(X)}')
print(f'Augmented train size : {len(X_augmented)}')

# ── Retrain XGBoost on augmented data ─────────────────
print('\n' + '='*50)
print('🚀 Retraining XGBoost with Pseudo-labels...')
print('='*50)

oof_xgb2  = np.zeros(len(X))
pred_xgb2 = np.zeros(len(X_test))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    # Val stays original, train gets pseudo-labels added
    X_tr_orig = X.iloc[tr_idx]
    y_tr_orig = y.iloc[tr_idx]
    X_val     = X.iloc[val_idx]
    y_val     = y.iloc[val_idx]

    # Add pseudo-labeled test data to train only
    X_tr_aug  = pd.concat([X_tr_orig, X_pseudo], axis=0, ignore_index=True)
    y_tr_aug  = pd.concat([y_tr_orig, y_pseudo],  axis=0, ignore_index=True)

    model = XGBClassifier(**best_xgb_params)
    model.fit(X_tr_aug, y_tr_aug,
              eval_set=[(X_val, y_val)],
              verbose=False)

    oof_xgb2[val_idx]  = model.predict_proba(X_val)[:, 1]
    pred_xgb2         += model.predict_proba(X_test)[:, 1] / N_SPLITS
    print(f'  Fold {fold+1:2d} | AUC: {roc_auc_score(y_val, oof_xgb2[val_idx]):.5f}')

print(f'\n📊 XGBoost+Pseudo OOF AUC: {roc_auc_score(y, oof_xgb2):.5f}')
print(f'📊 XGBoost original  AUC: {roc_auc_score(y, oof_xgb):.5f}')

# ── Retrain CatBoost on augmented data ────────────────
print('\n' + '='*50)
print('🚀 Retraining CatBoost with Pseudo-labels...')
print('='*50)

oof_cat2  = np.zeros(len(X))
pred_cat2 = np.zeros(len(X_test))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr_orig = X.iloc[tr_idx]
    y_tr_orig = y.iloc[tr_idx]
    X_val     = X.iloc[val_idx]
    y_val     = y.iloc[val_idx]

    X_tr_aug  = pd.concat([X_tr_orig, X_pseudo], axis=0, ignore_index=True)
    y_tr_aug  = pd.concat([y_tr_orig, y_pseudo],  axis=0, ignore_index=True)

    model = CatBoostClassifier(**best_cat_params)
    model.fit(X_tr_aug, y_tr_aug,
              eval_set              = (X_val, y_val),
              early_stopping_rounds = 100)

    oof_cat2[val_idx]  = model.predict_proba(X_val)[:, 1]
    pred_cat2         += model.predict_proba(X_test)[:, 1] / N_SPLITS
    print(f'  Fold {fold+1:2d} | AUC: {roc_auc_score(y_val, oof_cat2[val_idx]):.5f}')

print(f'\n📊 CatBoost+Pseudo OOF AUC: {roc_auc_score(y, oof_cat2):.5f}')
print(f'📊 CatBoost original  AUC: {roc_auc_score(y, oof_cat):.5f}')

# ── Summary ───────────────────────────────────────────
print('\n' + '='*50)
print('📊 Pseudo-label Summary')
print('='*50)
print(f'  XGBoost  original : {roc_auc_score(y, oof_xgb):.5f}')
print(f'  XGBoost  +pseudo  : {roc_auc_score(y, oof_xgb2):.5f}')
print(f'  CatBoost original : {roc_auc_score(y, oof_cat):.5f}')
print(f'  CatBoost +pseudo  : {roc_auc_score(y, oof_cat2):.5f}')

🔍 Generating pseudo-labels...
Total test samples      : 254655
High confidence samples : 141281
  → Pseudo churn=1      : 32767
  → Pseudo churn=0      : 108514

Original train size  : 601237
Augmented train size : 742518

🚀 Retraining XGBoost with Pseudo-labels...
  Fold  1 | AUC: 0.91550
  Fold  2 | AUC: 0.91615
  Fold  3 | AUC: 0.91838
  Fold  4 | AUC: 0.91442
  Fold  5 | AUC: 0.91492
  Fold  6 | AUC: 0.91366
  Fold  7 | AUC: 0.91533
  Fold  8 | AUC: 0.91530
  Fold  9 | AUC: 0.91637
  Fold 10 | AUC: 0.91633

📊 XGBoost+Pseudo OOF AUC: 0.91563
📊 XGBoost original  AUC: 0.91573

🚀 Retraining CatBoost with Pseudo-labels...


Default metric period is 5 because AUC is/are not implemented for GPU


  Fold  1 | AUC: 0.91536


Default metric period is 5 because AUC is/are not implemented for GPU


  Fold  2 | AUC: 0.91593


...

  Fold  8 | AUC: 0.91494


Default metric period is 5 because AUC is/are not implemented for GPU


  Fold  9 | AUC: 0.91610


Default metric period is 5 because AUC is/are not implemented for GPU


  Fold 10 | AUC: 0.91603

📊 CatBoost+Pseudo OOF AUC: 0.91533
📊 CatBoost original  AUC: 0.91538

📊 Pseudo-label Summary
  XGBoost  original : 0.91573
  XGBoost  +pseudo  : 0.91563
  CatBoost original : 0.91538
  CatBoost +pseudo  : 0.91533


In [7]:
from sklearn.linear_model import LogisticRegression
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── Use best predictions ──────────────────────────────
# Original beat pseudo so we use originals
# But blend ALL 5 model predictions for diversity

names       = ['XGBoost', 'LightGBM', 'CatBoost', 'XGB_pseudo', 'CAT_pseudo']
oof_matrix  = np.column_stack([oof_xgb, oof_lgb, oof_cat, oof_xgb2, oof_cat2])
pred_matrix = np.column_stack([pred_xgb, pred_lgb, pred_cat, pred_xgb2, pred_cat2])

# ── Optuna Weight Search ──────────────────────────────
print('🔍 Running Optuna weight search (2000 trials)...')

def objective(trial):
    w = np.array([trial.suggest_float(f'w{i}', 0.0, 1.0) for i in range(5)])
    w = w / w.sum()
    return roc_auc_score(y, (oof_matrix * w).sum(axis=1))

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=2000, show_progress_bar=True)

best_w = np.array([study.best_params[f'w{i}'] for i in range(5)])
best_w = best_w / best_w.sum()

print(f'\n⚖️  Optimal weights:')
for n, w in zip(names, best_w):
    print(f'  {n:15s}: {w:.4f}')

oof_blend  = (oof_matrix  * best_w).sum(axis=1)
pred_blend = (pred_matrix * best_w).sum(axis=1)
print(f'\n📊 Weighted Blend OOF AUC : {roc_auc_score(y, oof_blend):.5f}')

# ── Level-2 Stacker ───────────────────────────────────
print('\n🚀 Training Level-2 Stacker...')

oof_stack  = np.zeros(len(y))
pred_stack = np.zeros(len(X_test))

for fold, (tr_idx, val_idx) in enumerate(skf.split(oof_matrix, y)):
    meta = LogisticRegression(C=1.0, max_iter=1000)
    meta.fit(oof_matrix[tr_idx], y.iloc[tr_idx])
    oof_stack[val_idx]  = meta.predict_proba(oof_matrix[val_idx])[:, 1]
    pred_stack         += meta.predict_proba(pred_matrix)[:, 1] / N_SPLITS

print(f'📊 Level-2 Stack OOF AUC  : {roc_auc_score(y, oof_stack):.5f}')

# ── Pick best final prediction ────────────────────────
blend_auc = roc_auc_score(y, oof_blend)
stack_auc = roc_auc_score(y, oof_stack)
xgb_auc   = roc_auc_score(y, oof_xgb)

if blend_auc >= stack_auc and blend_auc >= xgb_auc:
    pred_final = pred_blend
    best_name  = f'Blend ({blend_auc:.5f})'
elif stack_auc >= xgb_auc:
    pred_final = pred_stack
    best_name  = f'Stack ({stack_auc:.5f})'
else:
    pred_final = pred_xgb
    best_name  = f'XGBoost only ({xgb_auc:.5f})'

print(f'\n✅ Using: {best_name}')

# ── Submit ────────────────────────────────────────────
sub['Churn'] = pred_final
sub.to_csv('submission.csv', index=False)

print(f'\n✅ submission.csv saved — {len(sub)} rows')
print(f'  Mean : {pred_final.mean():.4f}')
print(f'  Std  : {pred_final.std():.4f}')
print(f'  Min  : {pred_final.min():.4f}')
print(f'  Max  : {pred_final.max():.4f}')

# ── Final Summary ─────────────────────────────────────
print('\n' + '='*50)
print('🏆 FINAL SUMMARY')
print('='*50)
print(f'  XGBoost    OOF AUC : {roc_auc_score(y, oof_xgb):.5f}')
print(f'  LightGBM   OOF AUC : {roc_auc_score(y, oof_lgb):.5f}')
print(f'  CatBoost   OOF AUC : {roc_auc_score(y, oof_cat):.5f}')
print(f'  XGB+Pseudo OOF AUC : {roc_auc_score(y, oof_xgb2):.5f}')
print(f'  CAT+Pseudo OOF AUC : {roc_auc_score(y, oof_cat2):.5f}')
print(f'  Blend      OOF AUC : {blend_auc:.5f}')
print(f'  Stack      OOF AUC : {stack_auc:.5f}')
print(f'  ─────────────────────────────')
print(f'  BEST       OOF AUC : {max(blend_auc, stack_auc, xgb_auc):.5f} 🎯')
print('='*50)
print('\n✅ DONE — Output tab → submission.csv → Submit!')

🔍 Running Optuna weight search (2000 trials)...


  0%|          | 0/2000 [00:00<?, ?it/s]


⚖️  Optimal weights:
  XGBoost        : 0.5196
  LightGBM       : 0.0000
  CatBoost       : 0.1737
  XGB_pseudo     : 0.2410
  CAT_pseudo     : 0.0656

📊 Weighted Blend OOF AUC : 0.91582

🚀 Training Level-2 Stacker...
📊 Level-2 Stack OOF AUC  : 0.91569

✅ Using: Blend (0.91582)

✅ submission.csv saved — 254655 rows
  Mean : 0.3425
  Std  : 0.3493
  Min  : 0.0004
  Max  : 0.9952

🏆 FINAL SUMMARY
  XGBoost    OOF AUC : 0.91573
  LightGBM   OOF AUC : 0.91129
  CatBoost   OOF AUC : 0.91538
  XGB+Pseudo OOF AUC : 0.91563
  CAT+Pseudo OOF AUC : 0.91533
  Blend      OOF AUC : 0.91582
  Stack      OOF AUC : 0.91569
  ─────────────────────────────
  BEST       OOF AUC : 0.91582 🎯

✅ DONE — Output tab → submission.csv → Submit!


In [8]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.isotonic import IsotonicRegression

# ── Calibrate predictions to match true churn rate ───
print(f'Before calibration mean : {pred_final.mean():.4f}')
print(f'True churn rate         : {y.mean():.4f}')

# Isotonic calibration using OOF
iso = IsotonicRegression(out_of_bounds='clip')
iso.fit(oof_blend, y)
pred_calibrated = iso.predict(pred_blend)

print(f'After calibration mean  : {pred_calibrated.mean():.4f}')

# Save calibrated submission
sub['Churn'] = pred_calibrated
sub.to_csv('submission_calibrated.csv', index=False)

# Save original too just in case
sub['Churn'] = pred_final
sub.to_csv('submission_original.csv', index=False)

print(f'\n✅ Both saved:')
print(f'  submission_calibrated.csv — mean={pred_calibrated.mean():.4f}')
print(f'  submission_original.csv   — mean={pred_final.mean():.4f}')
print(f'\nSubmit BOTH and compare LB scores!')

Before calibration mean : 0.3425
True churn rate         : 0.2257
After calibration mean  : 0.2186

✅ Both saved:
  submission_calibrated.csv — mean=0.2186
  submission_original.csv   — mean=0.3425

Submit BOTH and compare LB scores!
